<a href="https://colab.research.google.com/github/Ryan56-56/project1ML/blob/main/bestsofar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import imageio.v2 as imageio
from tqdm import tqdm

# Config
BATCH_SIZE = 64
LR = 0.001
EPOCHS = 50
WEIGHT_DECAY = 1e-4

train_path = "/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/train"
test_path  = "/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/test"

CKPT_DIR = "/content/drive/MyDrive/CNN_model_CIFAR10"
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [25]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616]),
    T.RandomErasing(p=0.25)
])

test_transform = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616])
])


In [27]:
class CIFARFolderDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []
        self.classes = sorted(os.listdir(root))

        for label_idx, folder in enumerate(self.classes):
            folder_path = os.path.join(root, folder)
            for img_name in os.listdir(folder_path):
                self.samples.append((os.path.join(folder_path, img_name), label_idx))

        print(f"Found {len(self.samples)} images.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = imageio.imread(img_path)

        if self.transform:
            img = self.transform(img)

        return img, label


In [28]:
train_ds = CIFARFolderDataset(train_path, transform=train_transform)
test_ds  = CIFARFolderDataset(test_path, transform=test_transform)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds, batch_size=256, shuffle=False,
                      num_workers=2, pin_memory=True)


Found 50001 images.
Found 10000 images.


In [29]:
class BetterCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = BetterCNN().to(device)
model = torch.compile(model)


In [30]:
opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()


/tmp/ipykernel_3824/2638458450.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [31]:
def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for xb, yb in loop:
        xb = xb.to(device)
        yb = yb.to(device)

        opt.zero_grad()
        with torch.cuda.amp.autocast():
            preds = model(xb)
            loss = loss_fn(preds, yb)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} | Loss: {running_loss/len(train_dl):.4f}")


In [32]:
def evaluate():
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for xb, yb in test_dl:
            xb = xb.to(device)
            preds = model(xb).argmax(1).cpu()

            preds_all.extend(preds.numpy())
            labels_all.extend(yb.numpy())

    print("Accuracy:", accuracy_score(labels_all, preds_all))
    print("Precision:", precision_score(labels_all, preds_all, average='weighted'))
    print("Recall:", recall_score(labels_all, preds_all, average='weighted'))
    print("F1 Score:", f1_score(labels_all, preds_all, average='weighted'))


In [ ]:
best_f1 = 0.0

for epoch in range(EPOCHS):
    train_one_epoch(epoch)
    scheduler.step()

    if (epoch + 1) % 5 == 0:
        print("Evaluating...")
        evaluate()


Epoch 1/50:   0%|          | 0/782 [00:00<?, ?it/s]/tmp/ipykernel_3824/3494398000.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
